# 04 · Validate — train the ML success predictor + benchmark vs single-metric cutoffs

**Standard slot:** *validate (in silico).* **For Project 25 this is the core science (D3 part 2):**
train + cross-validate an ML model on the cohort feature table (`features → success`), report
**feature importance**, and benchmark its enrichment against the field's **single-metric cutoffs**
(scRMSD<2, pae_interaction<10, ...). This is the project's signature: closing the gap between
in-silico scores and experimental success by *learning* which features actually matter.

> **Honesty up front.** A real cohort is still a SMALL, BIASED, MULTI-TARGET dataset. We report
> **CV-AUC (mean ± std) and N**, never a single-split brag, and we keep the model small. The honest
> result is *"the model beats single-metric cutoffs by some margin"* — or, with small N, *"it doesn't
> yet"*. Both are valid capstone findings. **All numbers here are `EXAMPLE_DATA` (synthetic).**

Needs `results/cohort_table.csv` (from notebook 02). The ML part is **light / CPU-OK** — no GPU.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Load the cohort table + build (X, y)

`ml_predictor.build_cohort_table()` LOADS your real cohort table if present, else synthesizes the
deterministic `EXAMPLE_DATA` cohort. `features_and_label()` selects the complete feature columns and
drops incomplete rows. We print N and the class balance — the numbers that frame every claim below.

In [ ]:
import ml_predictor as ml
import pandas as pd, os

if os.path.exists("results/cohort_table.csv"):
    cohort = pd.read_csv("results/cohort_table.csv")
else:
    cohort = ml.build_cohort_table(seed=0)   # deterministic EXAMPLE_DATA fallback
    cohort.to_csv("results/cohort_table.csv", index=False)

X, y, feat_cols = ml.features_and_label(cohort)
print("cohort table:", cohort.shape, "| features used:", feat_cols)
print(f"N = {len(y)}  successes = {int(y.sum())}  base rate = {y.mean():.3f}  [EXAMPLE_DATA if synthetic]")
print("label origin:", cohort["label_origin"].value_counts().to_dict())

## 2 · Train + cross-validate the success predictor

`train_success_predictor()` fits a small model with **StratifiedKFold cross-validation** and returns
CV ROC-AUC (mean ± std), N, and notes. Default is interpretable LogisticRegression; `model="rf"`
(RandomForest) captures non-linear feature interactions; `model="xgb"` uses XGBoost if installed and
**falls back to RandomForest with a note** otherwise (so this runs anywhere).

In [ ]:
bundle = ml.train_success_predictor(X, y, feature_names=feat_cols, model="logreg", cv=5, seed=0)
print(f"model = {bundle['model_kind']}")
print(f"CV ROC-AUC = {bundle['cv_auc_mean']:.3f} +/- {bundle['cv_auc_std']:.3f}  "
      f"(folds={bundle['cv_folds']}, N={bundle['n']}, pos={bundle['n_pos']}, neg={bundle['n_neg']})")
for note in bundle["notes"]:
    print("  note:", note)

# Compare estimators honestly (RandomForest; XGBoost if available).
rf = ml.train_success_predictor(X, y, feature_names=feat_cols, model="rf", cv=5, seed=0)
xgb = ml.train_success_predictor(X, y, feature_names=feat_cols, model="xgb", cv=5, seed=0)
print(f"\nRandomForest CV-AUC = {rf['cv_auc_mean']:.3f} +/- {rf['cv_auc_std']:.3f}")
print(f"XGBoost/-fallback CV-AUC = {xgb['cv_auc_mean']:.3f} +/- {xgb['cv_auc_std']:.3f} ({xgb['model_kind']})")
print("\nAll AUCs are on EXAMPLE_DATA — they show the PLUMBING + honest reporting, not a real result.")

## 3 · Feature importance — which features actually carry the signal?

For LogisticRegression these are **standardized coefficients** (sign = direction: does a higher value
raise or lower predicted success?). For tree models they are gain/impurity importances (magnitude
only). This is the "which filters matter" answer the cohort cares about — read it against the
field-standard cutoffs.

In [ ]:
imp = ml.feature_importance(bundle)
print("feature importance (standardized LogReg coefficients; EXAMPLE_DATA):")
print(imp.to_string(index=False))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(imp["feature"][::-1], imp["importance"][::-1])
ax.set_xlabel("|importance|  (EXAMPLE_DATA)")
ax.set_title("Success-predictor feature importance")
plt.tight_layout(); plt.savefig("results/p25_feature_importance.png", dpi=150); plt.show()
print("saved results/p25_feature_importance.png")

## 4 · The headline benchmark — ML vs single-metric cutoffs (enrichment)

`compare_to_single_metric_cutoffs()` reports, for each field-standard cutoff **and** for the ML model
selecting a comparable fraction of designs: how many it selects, the **precision** (success rate among
selected), the **enrichment** (precision / base rate), and the **recall** (fraction of all successes
captured). The capstone question: *does the learned predictor enrich better — and/or recall more
successes — at a comparable selection size than any single metric?* Report what you find.

In [ ]:
cmp = ml.compare_to_single_metric_cutoffs(cohort, bundle)
print(f"base success rate = {cmp.attrs.get('base_rate')}  (N={cmp.attrs.get('n_total')}, "
      f"successes={cmp.attrs.get('n_success')})  [EXAMPLE_DATA]")
print()
print(cmp.to_string(index=False))
cmp.to_csv("results/p25_enrichment_vs_cutoffs.csv", index=False)

best_single = cmp[cmp["kind"] == "single-metric"]["enrichment"].max()
ml_row = cmp[cmp["kind"] == "ML-composite"]
ml_enr = float(ml_row["enrichment"].iloc[0]) if len(ml_row) else float("nan")
verdict = ("BEATS" if ml_enr >= best_single else "does NOT beat")
print(f"\nVERDICT (EXAMPLE_DATA): ML enrichment {ml_enr} {verdict} best single-metric enrichment {best_single}.")
print("On a real cohort this verdict may differ — REPORT IT HONESTLY, with N and CV-AUC.")

In [ ]:
# Enrichment bar chart (single metrics vs the ML composite). EXAMPLE_DATA.
fig, ax = plt.subplots(figsize=(7, 3.4))
colors = ["#888" if k == "single-metric" else "#1f77b4" for k in cmp["kind"]]
ax.bar(range(len(cmp)), cmp["enrichment"], color=colors)
ax.axhline(1.0, color="k", lw=0.8, ls="--", label="no enrichment (=base rate)")
ax.set_xticks(range(len(cmp)))
ax.set_xticklabels(cmp["selector"], rotation=40, ha="right", fontsize=7)
ax.set_ylabel("enrichment (precision / base rate)")
ax.set_title("ML predictor vs single-metric cutoffs (EXAMPLE_DATA)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/p25_enrichment.png", dpi=150); plt.show()
print("saved results/p25_enrichment.png")

## 5 · Cross-target generalization `[extension]`

A predictor that only works on the design type it was trained on is weak. Hold out one design type,
train on the rest, and test on the held-out type — the honest test of whether the learned rule
**generalizes** across the cohort. With small per-type N this is noisy; report the caveat.

In [ ]:
import numpy as np
print("leave-one-design-type-out CV-AUC (EXAMPLE_DATA; small-N, noisy):")
from sklearn.metrics import roc_auc_score
for held in sorted(cohort["design_type"].unique()):
    tr = cohort[cohort["design_type"] != held]
    te = cohort[cohort["design_type"] == held]
    Xtr, ytr, cols = ml.features_and_label(tr)
    b = ml.train_success_predictor(Xtr, ytr, feature_names=cols, model="logreg", seed=0)
    if b["estimator"] is None:
        print(f"  hold out {held:9s}: insufficient data"); continue
    Xte = te.dropna(subset=cols + ["success"])[cols].to_numpy(float)
    yte = te.dropna(subset=cols + ["success"])["success"].to_numpy(int)
    try:
        p = b["estimator"].predict_proba(Xte)[:, 1]
        auc = roc_auc_score(yte, p) if len(set(yte)) > 1 else float("nan")
    except Exception as e:  # noqa: BLE001
        auc = float("nan")
    print(f"  hold out {held:9s}: test ROC-AUC = {auc:.3f}  (test N={len(yte)})")
print("\nGeneralization across targets is the hard part; small per-type N makes this noisy (EXAMPLE_DATA).")

## D3 (part 2) checklist
- [ ] Cohort table loaded; **N + class balance** printed (frames every claim).
- [ ] Success predictor trained with **cross-validation**; CV-AUC (mean ± std) + N reported (not a single split).
- [ ] **Feature importance** computed + figure (`results/p25_feature_importance.png`).
- [ ] **Enrichment vs single-metric cutoffs** table + figure; honest VERDICT (beats / doesn't, by how much).
- [ ] (extension) Cross-target generalization (leave-one-type-out) reported with the small-N caveat.
- [ ] Every number flagged `EXAMPLE_DATA`; overfitting / N caveats stated.

**Next:** `05_validation_plan.ipynb` — execute-or-plan validation, integrate labels, honest hit-rate +
failure forensics + active-learning loop.